# 04 — TGN Training on 3W Dataset

**Goal:** Train and evaluate the Temporal Graph Network on the 3W oil well dataset.

## Key differences from Use Case 1 (synthetic data):
- Real industrial data with genuine noise and missing values
- Multiple wells as graph nodes (generalizable model)
- PRECEDES and CAUSALLY_COUPLED relations used by TGN
- Strict temporal split to prevent data leakage
- Weighted loss for class imbalance (rare events)

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import sys
sys.path.append('../../src')
from config import DATA_ROOT_3W, PROCESSED_DIR

torch.manual_seed(42)
np.random.seed(42)

PROCESSED_DIR = Path('../../data/processed')
MODELS_DIR    = Path('../../models')
MODELS_DIR.mkdir(exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


PyTorch: 2.9.1+cu128
Device: GPU


## 1. Load Data from Neo4j

In [2]:
from neo4j import GraphDatabase
from pathlib import Path
import pandas as pd

NEO4J_URI      = 'bolt://172.22.43.151:7687'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'your_password'

PROCESSED_DIR = Path('/home/obiaggi/3w_processed')
MODELS_DIR    = Path('/home/obiaggi/TKG_Thesis/models')
MODELS_DIR.mkdir(exist_ok=True)

# Carica tipi Normal + anomalie
dfs = []
# It was too big to be loaded for the TGN training
# for type_num in [0, 1, 3, 5]:
for type_num in [0, 3]:  # Solo Normal + Severe Slugging 
    f = PROCESSED_DIR / f'type_{type_num}.parquet'
    if f.exists():
        d = pd.read_parquet(f)
        # dfs.append(d)
        # need to reduc ethe number of lines for the knernel
        d = d.sample(n=50000, random_state=42)  # max 50k righe per tipo
        dfs.append(d)
        print(f'  Loaded type_{type_num}: {len(d):,} rows (sampled)')
       # print(f'  Loaded type_{type_num}: {len(d):,} rows')

df = pd.concat(dfs).reset_index(drop=True)
df['well_id'] = df['instance_id'].str.extract(r'(WELL-\d+)')
df['sensor_id'] = 'T-PDG'
df['value'] = df['T-PDG'].fillna(0)
df['is_anomaly'] = df['event_type'] > 0
df['timestamp'] = pd.RangeIndex(len(df))  # numerical index of the timestamp

print(f'\n✅ Total: {len(df):,} observations')
print(f'Anomaly rate: {df["is_anomaly"].mean()*100:.1f}%')

  Loaded type_0: 50,000 rows (sampled)
  Loaded type_3: 50,000 rows (sampled)

✅ Total: 100,000 observations
Anomaly rate: 50.0%


## 2. Prepare TGN Data

In [3]:
def prepare_tgn_data(df: pd.DataFrame):
    """
    Converts DataFrame to TGN event format.
    
    Split strategy: temporal (no random split to prevent leakage)
    Ref: Longa et al. [7] — future-transductive setting
    """
    if df.empty:
        return None, None, 0

    # Node index mapping
    wells   = df['well_id'].unique().tolist() if 'well_id' in df.columns else []
    sensors = df['sensor_id'].unique().tolist() if 'sensor_id' in df.columns else []
    all_nodes = wells + sensors
    node2idx  = {n: i for i, n in enumerate(all_nodes)}

    # Normalize values
    df = df.copy()
    scaler = StandardScaler()
    if 'value' in df.columns:
        df['value_norm'] = scaler.fit_transform(df[['value']].fillna(0))
    else:
        df['value_norm'] = 0

    # Temporal features
    df['ts_sec'] = pd.to_datetime(df['timestamp'] if 'timestamp' in df.columns else df['valid_from']).astype(np.int64) // 10**9
    df = df.sort_values('ts_sec')
    df['delta_t'] = df.groupby('sensor_id')['ts_sec'].diff().fillna(0)
    df['delta_t_norm'] = df['delta_t'] / (df['delta_t'].max() + 1e-8)

    # Temporal split 70/30
    split_idx = int(len(df) * 0.7)
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()

    def to_tensors(d):
        if 'well_id' in d.columns:
            src = torch.tensor([node2idx.get(w, 0) for w in d['well_id']], dtype=torch.long)
        else:
            src = torch.zeros(len(d), dtype=torch.long)
        dst = torch.tensor([node2idx.get(s, 0) for s in d['sensor_id']], dtype=torch.long)
        ef  = torch.tensor(d[['value_norm','delta_t_norm']].values, dtype=torch.float32)
        dt  = torch.tensor(d['delta_t_norm'].values, dtype=torch.float32).unsqueeze(1)
        y   = torch.tensor(d['is_anomaly'].fillna(False).astype(int).values, dtype=torch.float32)
        return src, dst, ef, dt, y

    print(f'Train: {len(train_df):,} events | Test: {len(test_df):,} events')
    print(f'Anomaly rate train: {train_df["is_anomaly"].mean()*100:.1f}%')
    print(f'Anomaly rate test:  {test_df["is_anomaly"].mean()*100:.1f}%')
    return to_tensors(train_df), to_tensors(test_df), len(all_nodes)

if not df.empty:
    train_data, test_data, num_nodes = prepare_tgn_data(df)
else:
    print('⚠️  Load data first')

Train: 70,000 events | Test: 30,000 events
Anomaly rate train: 71.4%
Anomaly rate test:  0.0%


## 3. Train TGN

In [4]:
# Import TGN from src/models
try:
    from models.tgn import TGN, train_tgn, evaluate_tgn
    print('✅ TGN imported from src/models')
except ImportError:
    print('⚠️  TGN not found in src/models — copy TGN-anomaly_detection.py there first')

if not df.empty and train_data is not None:
    model = TGN(
        num_nodes=num_nodes,
        memory_dim=64,    # larger than UC1 — more complex data
        message_dim=64,
        embed_dim=64,
        edge_dim=2,
    )
    print(f'TGN parameters: {sum(p.numel() for p in model.parameters()):,}')

    print('\n🏋️  Training TGN on 3W dataset...')
    train_tgn(model, train_data, epochs=5, batch_size=512)

    print('\n📊 Evaluation:')
    evaluate_tgn(model, test_data)

    # Save checkpoint
    checkpoint_path = MODELS_DIR / 'tgn_3w_checkpoint.pt'
    torch.save({
        'model_state': model.state_dict(),
        'num_nodes': num_nodes,
        'dataset': '3W',
        'epochs': 5,
    }, checkpoint_path)
    print(f'\n✅ Checkpoint saved: {checkpoint_path}')
else:
    print('⚠️  Load data first')

✅ TGN imported from src/models
TGN parameters: 65,537

🏋️  Training TGN on 3W dataset...
  Epoch 1/5 — loss: 1.7338
  Epoch 2/5 — loss: 1.1503
  Epoch 3/5 — loss: 2.2902
  Epoch 4/5 — loss: 0.5534
  Epoch 5/5 — loss: 0.8318

📊 Evaluation:

📈 Risultati: TGN (Temporal Graph Network)
              precision    recall  f1-score   support

      Normal      1.000     1.000     1.000     29998
     Anomaly      0.000     0.000     0.000         2

    accuracy                          1.000     30000
   macro avg      0.500     0.500     0.500     30000
weighted avg      1.000     1.000     1.000     30000

  AUC-ROC: 0.6108

✅ Checkpoint saved: /home/obiaggi/TKG_Thesis/models/tgn_3w_checkpoint.pt


/home/obiaggi/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/obiaggi/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/obiaggi/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
